<h2><center>Assignment 6</center></h2>
<h3><center>Programming for Data Science 2026</center></h3>
<b><center>Deadline: 14:00, April 16, 2026</center></b>

- The exercise will be marked as **passed** if you get **at least 10/16** points.

- Exercises must be handed in via **ILIAS** (Homework assignments). Submit your work as a **compressed (.zip)** file containing **one** `.py` **or** `.ipynb` file with **all exercises**.

- The name of **both** the `.zip` file and the `.py`/`.ipynb` file must be the *SurnameName* of the **two group members**, separated by an underscore.  
  Example: Tim Tabellen and Lara Lambda → `TabellenTim_LambdaLara.zip`  
  (The `.py`/`.ipynb` file must have the same name.)

- Use **comments** to explain your code and demonstrate that you understand the solutions and can discuss them.

- You are not expected to collaborate outside your group. Submitting other groups’ code as your own will result in **0 points**.

- For general questions about the lecture content, assignments, or the exam, please use the **ILIAS forum**.

- For individual questions about lecture content and the exam, contact: *roland.widmer@students.unibe.ch*  

- For individual questions about the exercises or grading, contact: *julien.brunner@students.unibe.ch* or *aline.steiner@students.unibe.ch*

#### Exercise 1: MultiIndex *(5 points)*

In this exercise you will create and manipulate a multi-index dataframe.

In [1]:
import pandas as pd

stores = [
    ('Länggasse', 'Mittelstrasse'),
    ('Länggasse', 'Falkenplatz'),
    ('Länggasse', 'Neufeld'),
    ('Bümpliz', 'Höhe'),
    ('Breitenrain',  'Breitenrainplatz'),
    ('Breitenrain',  'Kaserne'),
] # schema: region, name of the store

data = [
    [120, 135, 150, 160, 130, 140, 155, 170],
    [100, 110, 120, 130, 105, 115, 125, 140],
    [ 90,  95, 100, 105,  92,  98, 100, 105],
    [0, 0, 0, 255, 250, 260, 253, 230],
    [200, 210, 220, 230, 205, 215, 225, 240],
    [180, 190, 200, 210, 182, 192, 202, 215],
]

Our fictional vintage clothing company has six stores in Bern. One of the stores opened recently. For the years 2024 and 2025, the quarterly sales are stored in the matrix `data` (each row represents one store and each column represents one quarter). Note that the sales of the Höhe store are 0 for 2024 Q1–Q3 because the store opened in 2024 Q4.

1. Create a MultiIndex using `stores` and `pd.MultiIndex.from_tuples`. Name the index levels `region` and `store`.

In [2]:
index = pd.MultiIndex.from_tuples(stores, names=['region', 'store'])

2. Create the columns of our dataframe by calculating all possible combinations of years and quarters. Use the appropriate method offered by pandas.

In [3]:
years = [2024, 2025]
quarters = ['Q1', 'Q2', 'Q3', 'Q4']

columns = pd.MultiIndex.from_product([years, quarters], names=['year', 'quarter'])
columns

MultiIndex([(2024, 'Q1'),
            (2024, 'Q2'),
            (2024, 'Q3'),
            (2024, 'Q4'),
            (2025, 'Q1'),
            (2025, 'Q2'),
            (2025, 'Q3'),
            (2025, 'Q4')],
           names=['year', 'quarter'])

3. Create a pandas DataFrame `sales_df` using `data` and the index and columns created above.

In [4]:
sales_df = pd.DataFrame(data, index=index, columns=columns)
sales_df

year                         2024                2025               
quarter                        Q1   Q2   Q3   Q4   Q1   Q2   Q3   Q4
region      store                                                   
Länggasse   Mittelstrasse     120  135  150  160  130  140  155  170
            Falkenplatz       100  110  120  130  105  115  125  140
            Neufeld            90   95  100  105   92   98  100  105
Bümpliz     Höhe                0    0    0  255  250  260  253  230
Breitenrain Breitenrainplatz  200  210  220  230  205  215  225  240
            Kaserne           180  190  200  210  182  192  202  215

4. The management would like to discuss the future of the Neufeld store which performs the worst among the stores in Länggasse. Print the sales of all stores in Länggasse in 2025 twice: once using `loc` and once using `xs`.

In [5]:
print(sales_df.loc['Länggasse', 2025])
print("=====")
print(sales_df.xs('Länggasse', level='region')[2025])

quarter         Q1   Q2   Q3   Q4
store                            
Mittelstrasse  130  140  155  170
Falkenplatz    105  115  125  140
Neufeld         92   98  100  105
=====
quarter         Q1   Q2   Q3   Q4
store                            
Mittelstrasse  130  140  155  170
Falkenplatz    105  115  125  140
Neufeld         92   98  100  105


5. Next, we are interested in the average quarterly sales in 2024 and want to compare them to the average quarterly sales in 2025.

- The 0 values for the Höhe store (which only opened in 2024 Q4) bias the quarterly average. Treat the 0 values for the Höhe store in 2024 Q1–Q3 as missing values when computing the 2024 average for that store.
- For every store, calculate the difference between quarterly average sales of 2024 and 2025.

Hint: Replacing the zeros with missing values might be helpful.

In [6]:
sales_df_cleaned = sales_df.replace(0, pd.NA)
avg_2024 = sales_df_cleaned[2024].mean(axis=1)
avg_2025 = sales_df_cleaned[2025].mean(axis=1)

# Calculate the difference
diff = avg_2025 - avg_2024

print("--- Difference in Average Quarterly Sales (2025 - 2024) ---")
print(diff)

--- Difference in Average Quarterly Sales (2025 - 2024) ---
region       store           
Länggasse    Mittelstrasse        7.5
             Falkenplatz         6.25
             Neufeld             1.25
Bümpliz      Höhe               -6.75
Breitenrain  Breitenrainplatz    6.25
             Kaserne             2.75
dtype: object


#### Exercise 2: Merging DataFrames *(3 points)*

We start by defining three DataFrames:

In [7]:
# Series
series_df = pd.DataFrame({
    'series_id': [1, 2, 3],
    'title': ['Stranger Things', 'The Office', 'Breaking Bad'],
    'genre': ['Sci-Fi', 'Comedy', 'Drama']
})

# Episodes
episodes_df = pd.DataFrame({
    'episode_id': [101, 102, 103, 201, 202, 301],
    'series_id': [1, 1, 1, 2, 2, 3],
    'season': [1, 1, 1, 1, 1, 1],
    'episode_number': [1, 2, 3, 1, 2, 1],
    'title': [
        'Chapter One', 'Chapter Two', 'Chapter Three',
        'Pilot', 'Diversity Day',
        'Pilot'
    ]
})

# Actors
actors_df = pd.DataFrame({
    'episode_id': [101, 101, 102, 103, 201, 202, 999],
    'actor': [
        'Millie Bobby Brown', 'Finn Wolfhard',
        'Millie Bobby Brown', 'Noah Schnapp',
        'Steve Carell', 'John Krasinski',
        'Unknown Actor'
    ]
})

1. Merge `episodes_df` with `series_df` to attach the series title and genre to each episode. Store the result as `episodes_full_df` with additional columns `series_title` and `genre`.  

In [8]:
episodes_full_df = episodes_df.merge(
    series_df.rename(columns={'title': 'series_title'}),
    on='series_id',
    how='left'
)
episodes_full_df

,episode_id,series_id,season,episode_number,title,series_title,genre
0,101,1,1,1,Chapter One,Stranger Things,Sci-Fi
1,102,1,1,2,Chapter Two,Stranger Things,Sci-Fi
2,103,1,1,3,Chapter Three,Stranger Things,Sci-Fi
3,201,2,1,1,Pilot,The Office,Comedy
4,202,2,1,2,Diversity Day,The Office,Comedy
5,301,3,1,1,Pilot,Breaking Bad,Drama


2. Merge `episodes_df` with `actors_df` and store the result in `episode_actors_df`. Explain the difference between `how="inner"` and `how="left"` in this example, especially which rows are kept or discarded in each case.

In [9]:
episode_actors_df_left = episodes_df.merge(actors_df, on='episode_id', how='left')
episode_actors_df_inner = episodes_df.merge(actors_df, on='episode_id', how='inner')
print(episode_actors_df_left.head())
print(episode_actors_df_left.head())

   episode_id  series_id  season  episode_number          title  \
0         101          1       1               1    Chapter One   
1         101          1       1               1    Chapter One   
2         102          1       1               2    Chapter Two   
3         103          1       1               3  Chapter Three   
4         201          2       1               1          Pilot   

                actor  
0  Millie Bobby Brown  
1       Finn Wolfhard  
2  Millie Bobby Brown  
3        Noah Schnapp  
4        Steve Carell  
   episode_id  series_id  season  episode_number          title  \
0         101          1       1               1    Chapter One   
1         101          1       1               1    Chapter One   
2         102          1       1               2    Chapter Two   
3         103          1       1               3  Chapter Three   
4         201          2       1               1          Pilot   

                actor  
0  Millie Bobby Brown  
1 

**Your answer**: Similarly to SQL queries, inner will keeps only rows where both keys exists (eg. episode_id). Left with keep all the records from the left table and add columns of the right table if keys match (or nan instead)

3. Create a dataframe `casting_df` with columns `episode_title`, `episode_number`, `season`, `series_title`, `actor` by combining the data from all three tables. Rename and reorder the columns where necessary. The final dataframe should contain all episodes, even if no actor is listed for them. Actor records with invalid episode_id values should be excluded.

In [10]:
casting_df = (
    episodes_df.merge(series_df.rename(columns={'title': 'series_title'}), on='series_id', how='left')
    .merge(actors_df, on='episode_id', how='left')
    .rename(columns={'title': 'episode_title'})
)

casting_df = casting_df[['episode_title', 'episode_number', 'season', 'series_title', 'actor']]
casting_df

,episode_title,episode_number,season,series_title,actor
0,Chapter One,1,1,Stranger Things,Millie Bobby Brown
1,Chapter One,1,1,Stranger Things,Finn Wolfhard
2,Chapter Two,2,1,Stranger Things,Millie Bobby Brown
3,Chapter Three,3,1,Stranger Things,Noah Schnapp
4,Pilot,1,1,The Office,Steve Carell
5,Diversity Day,2,1,The Office,John Krasinski
6,Pilot,1,1,Breaking Bad,NaN


#### Exercise 3: Gas prices *(5 points)*

In this exercise, we look at fuel prices in the UK.

Source: [fuelcosts.co.uk](https://www.kaggle.com/datasets/jamesb7/fuel-prices-uk?resource=download)

1. Using the files `data/csv/price_history.csv` and `data/csv/stations.csv`, create a merged dataframe `gas_prices_df`.

In [11]:
gas_prices_df = pd.read_csv("../data/csv/price_history.csv").merge(pd.read_csv("../data/csv/stations.csv"), on='node_id')

2. For the gas station with the cheapest petrol (E10) and the gas station with the cheapest diesel (B7_STANDARD) recorded on 30 March 2026, print the name (`trading_name`, `brand_name`, `organisation_name`) and the address (`address_line_1`, `city`, `county`, `postcode`).

Hint: To keep only rows recorded on 2026-03-30, you may use:

```python
days = pd.to_datetime(["2026-03-30"], utc=True)

gas_prices_df["recorded_at"] = pd.to_datetime(gas_prices_df["recorded_at"]).dt.normalize() # convert the column recorded_at to datetime and remove hours

filtered_gas_prices_df = gas_prices_df[
    gas_prices_df["recorded_at"].dt.isin(days)
]
```

In [12]:
days = pd.to_datetime(["2026-03-30"], utc=True)
gas_prices_df["recorded_at"] = pd.to_datetime(gas_prices_df["recorded_at"]).dt.normalize()
filtered_gas_prices_df = gas_prices_df[gas_prices_df["recorded_at"].isin(days)]

info_cols = ['trading_name', 'brand_name', 'organisation_name', 'address_line_1', 'city', 'county', 'postcode']

e10_df = filtered_gas_prices_df[filtered_gas_prices_df['fuel_type'] == 'E10']
cheapest_e10_idx = e10_df['price_pence'].idxmin()
cheapest_e10 = e10_df.loc[cheapest_e10_idx]

b7_df = filtered_gas_prices_df[filtered_gas_prices_df['fuel_type'] == 'B7_STANDARD']
cheapest_b7_idx = b7_df['price_pence'].idxmin()
cheapest_b7 = b7_df.loc[cheapest_b7_idx]

print("Cheapest E10 Gas Station :")
print(cheapest_e10[info_cols])

print("Cheapest B7_STANDARD Gas Station :")
print(cheapest_b7[info_cols])

Cheapest E10 Gas Station :
trading_name         KINGSWESTON SERVICE STATION
brand_name                                Valero
organisation_name                            NaN
address_line_1                   KINGSWESTON AVE
city                                SHIREHAMPTON
county                                   BRISTOL
postcode                                BS11 0AH
Name: 9047, dtype: object
Cheapest B7_STANDARD Gas Station :
trading_name         ASDA TOTTENHAM WHITE HART LANE PETROL FILLING ...
brand_name                                                        Asda
organisation_name                                                  NaN
address_line_1                      335-337 WHITE HART LANE, TOTTENHAM
city                                                            LONDON
county                                                  GREATER LONDON
postcode                                                       N17 7LY
Name: 9468, dtype: object


3. Set a MultiIndex using node_id, fuel_type, and recorded_at as index levels. Get all recorded prices for fuel type `E10` on the three days 2026-02-27, 2026-03-15, and 2026-03-30 using `.loc` or `.xs` and print the average price for the three days.

In [13]:
days = pd.to_datetime(["2026-02-27", "2026-03-15", "2026-03-30"], utc=True)

gas_prices_multi_df = gas_prices_df.set_index(['node_id', 'fuel_type', 'recorded_at'])

for day in days:
    day_prices = gas_prices_multi_df.xs(('E10', day), level=('fuel_type', 'recorded_at'))
    avg_price = day_prices['price_pence'].mean()

    print(f"Average E10 price on {day.date()}: {avg_price:.2f}")

Average E10 price on 2026-02-27: 137.72
Average E10 price on 2026-03-15: 148.32
Average E10 price on 2026-03-30: 152.64


4. Compare the average price of E10 fuel on 2026-03-30 between motorway and non-motorway stations.

In [14]:
e10_march_30 = filtered_gas_prices_df[filtered_gas_prices_df['fuel_type'] == 'E10']

motorway_comparison = e10_march_30.groupby('is_motorway')['price_pence'].mean()

print("Average E10 price on 2026-03-30 :")
print(motorway_comparison)

Average E10 price on 2026-03-30 :
is_motorway
False    152.214775
True     169.643662
Name: price_pence, dtype: float64


5. Compare the average E10 price change over the last month for the following four brands: Esso, BP, Shell, and Sainsbury’s. Iterate through the four brands and calculate a subset dataframe containing only the rows of that brand. For every brand, calculate the difference of the mean E10 price on 2026-02-27 and on 2026-03-30. Print the price difference.

In [15]:
brands = ["Esso", "Bp", "Shell", "Sainsbury'S"]
day_1 = pd.to_datetime("2026-02-27", utc=True)
day_2 = pd.to_datetime("2026-03-30", utc=True)

e10_all = gas_prices_df[gas_prices_df['fuel_type'] == 'E10']

print("E10 Price Change (2026-03-30 vs 2026-02-27)")
for brand in brands:
    brand_df = e10_all[e10_all['brand_name'].str.upper() == brand.upper()]

    mean_day_1 = brand_df[brand_df['recorded_at'] == day_1]['price_pence'].mean()
    mean_day_2 = brand_df[brand_df['recorded_at'] == day_2]['price_pence'].mean()

    price_diff = mean_day_2 - mean_day_1
    print(f"{brand}: {price_diff:+.2f}")

E10 Price Change (2026-03-30 vs 2026-02-27)
Esso: +20.58
Bp: +20.64
Shell: +16.73
Sainsbury'S: +18.22


#### Exercise 4: Fitbit *(3 points)*

In this exercise, we work with a Fitbit data set.

Source: [https://www.kaggle.com/datasets/arashnic/fitbit](https://www.kaggle.com/datasets/arashnic/fitbit)

Unfortunately, some values are missing in the `TotalSteps` and `Calories` columns.

1. Read `data/csv/fitbit.csv` and store it in a pandas DataFrame `fitbit_df`. Print the number of missing values in the columns `Calories` and `TotalSteps`.

In [16]:
fitbit_df = pd.read_csv("../data/csv/fitbit.csv")

missing_calories = fitbit_df['Calories'].isna().sum()
missing_steps = fitbit_df['TotalSteps'].isna().sum()

print(f"Missing values in Calories: {missing_calories}")
print(f"Missing values in TotalSteps: {missing_steps}")

Missing values in Calories: 43
Missing values in TotalSteps: 68


2. Let us assume that there is a direct linear relationship between calories and total steps:

    $n_{calories} = n_{steps} \cdot c, $
    
    $n_{steps} = n_{calories} \cdot s, $

    where $c$ denotes calories per step and $s=\frac{1}{c}$ steps per calorie.

    Estimate $c$ by calculating the average calories divided by the average total steps.

In [17]:
avg_calories = fitbit_df['Calories'].mean()
avg_total_steps = fitbit_df['TotalSteps'].mean()

c = avg_calories / avg_total_steps
print(f"Estimated calories per step : {c:.4f}")

Estimated calories per step : 0.2933


3. Fill the missing values using our estimation technique. Can this approach fill all missing values? Explain the result.

In [18]:
s = 1 / c

fitbit_df['Calories'] = fitbit_df['Calories'].fillna(fitbit_df['TotalSteps'] * c)
fitbit_df['TotalSteps'] = fitbit_df['TotalSteps'].fillna(fitbit_df['Calories'] * s)

print(f"Remaining missing in Calories: {fitbit_df['Calories'].isna().sum()}")
print(f"Remaining missing in TotalSteps: {fitbit_df['TotalSteps'].isna().sum()}")
# Remaining missing values are entries with 0 values

Remaining missing in Calories: 13
Remaining missing in TotalSteps: 13
